In [6]:
import torch
from mint.helpers.extract import load_config, CSVDataset, CollateFn, MINTWrapper

ModuleNotFoundError: No module named 'mint'

In [ ]:
cfg = load_config("../mint/data/esm2_t33_650M_UR50D.json") # model config
device = 'cuda:0' # GPU device
checkpoint_path = '../model_checkpoints/mint_base/mint.ckpt' # Where you stored the model checkpoint

dataset = CSVDataset('../mint/data/protein_sequences.csv', 'Protein_Sequence_1', 'Protein_Sequence_2')
loader = torch.utils.data.DataLoader(dataset, batch_size=2, collate_fn=CollateFn(512), shuffle=False) 

wrapper = MINTWrapper(cfg, checkpoint_path, device=device)

chains, chain_ids = next(iter(loader)) # Get the first batch
chains = chains.to(device)
chain_ids = chain_ids.to(device)
embeddings = wrapper(chains, chain_ids)  # Generate embeddings
print(embeddings.shape)

UnpicklingError: Weights only load failed. This file can still be loaded, to do so you have two options, [1mdo those steps only if you trust the source of the checkpoint[0m. 
	(1) In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code execution. Do it only if you got the file from a trusted source.
	(2) Alternatively, to load with `weights_only=True` please check the recommended steps in the following error message.
	WeightsUnpickler error: Unsupported global: GLOBAL argparse.Namespace was not an allowed global by default. Please use `torch.serialization.add_safe_globals([argparse.Namespace])` or the `torch.serialization.safe_globals([argparse.Namespace])` context manager to allowlist this global if you trust this class/function.

Check the documentation of torch.load to learn more about types accepted by defa

In [ ]:
%%bash
# Navigate to the folder containing your scripts (update if needed)
cd /home/oscar/evolutionary_simulations/mint/downstream/GeneralPPI

# Run the Feature Extraction
python embeddings_mint.py \
  --task SKEMPI \
  --model_name mint \
  --checkpoint_path /home/oscar/evolutionary_simulations/model_checkpoints/mint_base/mint.ckpt \
  --devices 0 \
  --bs 1 \

Process is terminated.


In [ ]:
%%bash
cd /home/oscar/evolutionary_simulations/mint/downstream/GeneralPPI
python finetune_general.py --task SKEMPI --model mint --use_mlp_for_cv